# BÀI TIỂU LUẬN / ĐỒ ÁN MÔN HỌC: PHÁT TRIỂN HỆ THỐNG THÔNG MINH
## Đề tài: Hệ Đa Tác Tử Kết Hợp Học Máy, Deep Graph và Q-Learning Sàng Lọc & Quản Lý Bệnh Tiểu Đường
**Sinh viên:** Nguyễn Tự Kiên (B23DCCN465) | CT01 — Nhóm 07

### 1. Khởi tạo Môi trường & Nạp Tập Dữ Liệu 150.000 Mẫu Dịch Tễ Việt Nam

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập đường dẫn import module src
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data_preprocessing import preprocess_tabular_data
X_train, X_test, y_train, y_test, feature_cols = preprocess_tabular_data()
print(f"Tập Train: {X_train.shape}, Tập Test: {X_test.shape}")

### 2. Thực Nghiệm Chương 2: Huấn luyện Mô hình Tabular ML & Đánh giá ROC-AUC

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

clf = HistGradientBoostingClassifier(max_iter=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, target_names=['Không ĐTĐ', 'Mắc ĐTĐ']))

### 3. Thực Nghiệm Chương 5 & 6: Medical Knowledge Graph và Deep Graph GCN

In [ ]:
from src.knowledge.knowledge_graph import MedicalKnowledgeGraph
from src.models.deep_graph import DeepGraphComorbidityGNN

kg = MedicalKnowledgeGraph()
gnn = DeepGraphComorbidityGNN()

sample_patient = {'BMI': 28.5, 'Glucose': 145.0, 'HbA1c': 7.2, 'HighBP': 1, 'Age': 8}
kg_res = kg.query_graph_rag(sample_patient)
gnn_res = gnn.forward(sample_patient)

print("=== GRAPH RAG MATCHED TRIPLES ===")
for t in kg_res['matched_triples']:
    print(f"- {t['source']} --({t['relation']})--> {t['target']}")

print("\n=== DEEP GRAPH GCN COMORBIDITY PREDICTION ===")
for c in gnn_res['predicted_comorbidities'][:3]:
    print(f"- Bệnh đồng mắc: {c['comorbidity']} | Rủi ro: {c['risk_score']*100:.1f}%")

### 4. Thực Nghiệm Q-Learning: Tối ưu Phác đồ Điều trị Tuần tự

In [ ]:
from src.models.qlearning_treatment import DiabetesQLearningEngine
rl = DiabetesQLearningEngine()
rx_res = rl.recommend_treatment(sample_patient)
print(f"Trạng thái State ID: {rx_res['state_id']}")
print(f"Hành động tối ưu (Q-Learning): {rx_res['recommended_action']}")
print(f"Chi tiết can thiệp: {rx_res['intervention_details']}")

### 5. Thực Nghiệm Chương 7: Hội chẩn Đa Tác Tử (Vòng tròn lớn trên bảng + / -)

In [ ]:
from src.agents.orchestrator import OrchestratorCoordinatorAgent
orchestrator = OrchestratorCoordinatorAgent()
full_res = orchestrator.run_full_consultation(sample_patient)
print(full_res['narrative_response'])